In [53]:
%pip install pandas numpy torch scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [54]:
import pandas as pd
import numpy as np
import torch
import random
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold

In [55]:
df = pd.read_csv('Datasets/UCL/uci.csv')

df.drop(columns=['FILENAME', 'Domain', 'TLD', 'Title'], inplace=True)
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

pd.set_option('display.max_columns', None)
df.head()


,URL,URLLength,DomainLength,IsDomainIP,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,URLCharProb,TLDLength,NoOfSubDomain,HasObfuscation,NoOfObfuscatedChar,ObfuscationRatio,NoOfLettersInURL,LetterRatioInURL,NoOfDegitsInURL,DegitRatioInURL,NoOfEqualsInURL,NoOfQMarkInURL,NoOfAmpersandInURL,NoOfOtherSpecialCharsInURL,SpacialCharRatioInURL,IsHTTPS,LineOfCode,LargestLineLength,HasTitle,DomainTitleMatchScore,URLTitleMatchScore,HasFavicon,Robots,IsResponsive,NoOfURLRedirect,NoOfSelfRedirect,HasDescription,NoOfPopup,NoOfiFrame,HasExternalFormSubmit,HasSocialNet,HasSubmitButton,HasHiddenFields,HasPasswordField,Bank,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,https://www.southbankmosaics.com,31,24,0,100.0,1.000000,0.522907,0.061933,3,1,0,0,0.0,18,0.581,0,0.0,0,0,0,1,0.032,1,558,9381,1,0.000000,0.000000,0,1,1,0,0,0,0,1,0,0,1,1,0,1,0,0,1,34,20,28,119,0,124,1
1,https://www.uni-mainz.de,23,16,0,100.0,0.666667,0.032650,0.050207,2,1,0,0,0.0,9,0.391,0,0.0,0,0,0,2,0.087,1,618,9381,1,55.555556,55.555556,1,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0,1,50,9,8,39,0,217,1
2,https://www.voicefmradio.co.uk,29,22,0,100.0,0.866667,0.028555,0.064129,2,2,0,0,0.0,15,0.517,0,0.0,0,0,0,2,0.069,1,467,682,1,46.666667,46.666667,0,1,1,0,0,1,0,0,0,0,1,1,0,0,0,0,1,10,2,7,42,2,5,1
3,https://www.sfnmjournal.com,26,19,0,100.0,1.000000,0.522907,0.057606,3,1,0,0,0.0,13,0.500,0,0.0,0,0,0,1,0.038,1,6356,26824,1,0.000000,0.000000,0,1,1,0,0,0,1,12,0,1,1,1,0,0,1,1,1,3,27,15,22,1,31,1
4,https://www.rewildingargentina.org,33,26,0,100.0,1.000000,0.079963,0.059441,3,1,0,0,0.0,20,0.606,0,0.0,0,0,0,1,0.030,1,6089,28404,1,100.000000,100.000000,0,1,1,1,1,1,0,2,0,1,1,1,0,1,1,0,1,244,15,34,72,1,85,1


In [56]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# URL Preprocessing

class URLPreprocessor:
    def __init__(self, max_len=200):
        self.max_len = max_len
        self.char2idx = None
        self.idx2char = None

    def fit(self, urls):
        # Build character vocabulary
        all_text = ''.join(urls)
        vocab = sorted(list(set(all_text)))
        self.char2idx = {ch: idx+1 for idx, ch in enumerate(vocab)}  # 0 reserved for padding
        self.idx2char = {idx: ch for ch, idx in self.char2idx.items()}

    def url_to_seq(self, url):
        return [self.char2idx[ch] for ch in url if ch in self.char2idx]

    def pad_or_truncate(self, seq):
        seq = torch.tensor(seq, dtype=torch.long)
        if len(seq) > self.max_len:
            return seq[:self.max_len]
        else:
            padding = torch.zeros(self.max_len - len(seq), dtype=torch.long)
            return torch.cat([seq, padding])

    def transform(self, urls):
        sequences = [self.pad_or_truncate(self.url_to_seq(url)) for url in urls]
        return torch.stack(sequences)  # shape: (num_samples, max_len)
    

# Numerical Feature Preprocessing

class NumericPreprocessor:
    def __init__(self):
        self.scaler = StandardScaler()

    def fit(self, X_numeric):
        self.scaler.fit(X_numeric)

    def transform(self, X_numeric):
        X_scaled = self.scaler.transform(X_numeric)
        return torch.tensor(X_scaled, dtype=torch.float)
    
    

def preprocess_df(df, url_col='URL', label_col='label'):
    # Separate features
    numeric_cols = [col for col in df.columns if col != url_col and col != label_col]
    X_numeric = df[numeric_cols].values
    y = torch.tensor(df[label_col].values, dtype=torch.long)

    # Preprocess URLs
    url_processor = URLPreprocessor(max_len=200)
    url_processor.fit(df[url_col].tolist())
    X_url = url_processor.transform(df[url_col].tolist())

    # Preprocess numeric features
    numeric_processor = NumericPreprocessor()
    numeric_processor.fit(X_numeric)
    X_numeric_scaled = numeric_processor.transform(X_numeric)

    return X_url, X_numeric_scaled, y, url_processor, numeric_processor



In [57]:
X_url, X_numeric, y, url_proc, num_proc = preprocess_df(df)
print(X_url.shape, X_numeric.shape, y.shape)

torch.Size([235795, 200]) torch.Size([235795, 50]) torch.Size([235795])


In [58]:
# CNN-LSTM Model

class CNNLSTMPhishing(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_numeric_features,
                 cnn_out_channels=64, cnn_kernel_size=5, lstm_hidden_size=128,
                 lstm_num_layers=1, num_classes=2, dropout=0.5):
        super(CNNLSTMPhishing, self).__init__()

        # -------------------
        # Embedding layer for URL sequences
        # -------------------
        self.embedding = nn.Embedding(num_embeddings=vocab_size,
                                      embedding_dim=embedding_dim,
                                      padding_idx=0)
        
        # -------------------
        # 1D CNN to capture local patterns in URLs
        # -------------------
        self.conv1 = nn.Conv1d(in_channels=embedding_dim,
                               out_channels=cnn_out_channels,
                               kernel_size=cnn_kernel_size,
                               padding=cnn_kernel_size//2)
        
        # -------------------
        # LSTM to capture sequential dependencies
        # -------------------
        self.lstm = nn.LSTM(input_size=cnn_out_channels,
                            hidden_size=lstm_hidden_size,
                            num_layers=lstm_num_layers,
                            batch_first=True,
                            bidirectional=True)

        # -------------------
        # Dropout layer for regularization
        # -------------------
        self.dropout = nn.Dropout(dropout)

        # -------------------
        # Fully connected layer to combine LSTM output + numeric features
        # -------------------
        self.fc = nn.Linear(lstm_hidden_size*2 + num_numeric_features, num_classes)

    def forward(self, x_url, x_numeric):
        # x_url: (batch_size, seq_len)
        # x_numeric: (batch_size, num_numeric_features)

        # -------------------
        # Embed URL characters
        # -------------------
        x = self.embedding(x_url)  # (batch_size, seq_len, embedding_dim)

        # -------------------
        # CNN expects (batch_size, channels, seq_len)
        # -------------------
        x = x.permute(0, 2, 1)  # (batch_size, embedding_dim, seq_len)
        x = F.relu(self.conv1(x))  # (batch_size, cnn_out_channels, seq_len)
        x = x.permute(0, 2, 1)  # (batch_size, seq_len, cnn_out_channels)

        # -------------------
        # Pass through LSTM
        # -------------------
        lstm_out, (h_n, c_n) = self.lstm(x)
        # Use the last hidden state from both directions
        lstm_out = torch.cat((h_n[-2], h_n[-1]), dim=1)  # (batch_size, lstm_hidden_size*2)

        # -------------------
        # Concatenate numeric features
        # -------------------
        combined = torch.cat([lstm_out, x_numeric], dim=1)  # (batch_size, lstm_hidden_size*2 + num_numeric_features)
        combined = self.dropout(combined)

        # -------------------
        # Final classification
        # -------------------
        out = self.fc(combined)  # (batch_size, num_classes)
        return out

In [59]:
X_url_train, X_url_test, X_num_train, X_num_test, y_train, y_test = train_test_split(
    X_url, X_numeric, y, test_size=0.2, random_state=42, stratify=y
)

# Convert numpy → torch tensors
train_dataset = TensorDataset(
    X_url_train, X_num_train, y_train
)
test_dataset = TensorDataset(
    X_url_test, X_num_test, y_test
)

# DataLoader wraps data for batching + shuffling
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

# Hyperparameters
vocab_size = len(url_proc.char2idx) + 1
embedding_dim = 50
num_numeric_features = X_numeric.shape[1]
num_classes = 2

# Model
model = CNNLSTMPhishing(vocab_size, embedding_dim, num_numeric_features, num_classes=num_classes)

# Loss function (for classification)
criterion = nn.CrossEntropyLoss()

# Optimizer (Adam works well for text)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for x_url_batch, x_num_batch, y_batch in train_loader:
        # Reset gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(x_url_batch, x_num_batch)
        loss = criterion(outputs, y_batch)

        # Backward pass + optimize
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

Epoch [1/5], Loss: 0.0158
Epoch [2/5], Loss: 0.0063
Epoch [3/5], Loss: 0.0035
Epoch [4/5], Loss: 0.0036
Epoch [5/5], Loss: 0.0034


In [ ]:
# model.eval()
# correct, total = 0, 0

# with torch.no_grad():  # No gradient tracking
#     for x_url_batch, x_num_batch, y_batch in test_loader:
#         outputs = model(x_url_batch, x_num_batch)
#         preds = torch.argmax(outputs, dim=1)
#         correct += (preds == y_batch).sum().item()
#         total += y_batch.size(0)

# print(f"Test Accuracy: {100 * correct / total:.2f}%")

from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():  # No gradient tracking
    for x_url_batch, x_num_batch, y_batch in test_loader:
        outputs = model(x_url_batch, x_num_batch)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())

# Convert to numpy arrays
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Compute metrics
accuracy = (all_preds == all_labels).mean() * 100
precision = precision_score(all_labels, all_preds, zero_division=0)
recall = recall_score(all_labels, all_preds, zero_division=0)
f1 = f1_score(all_labels, all_preds, zero_division=0)
conf_matrix = confusion_matrix(all_labels, all_preds)


# Print results
print(f"Test Accuracy : {accuracy:.2f}%")
print(f"Precision     : {precision:.4f}")
print(f"Recall        : {recall:.4f}")
print(f"F1-Score      : {f1:.4f}")
print("Confusion Matrix:\n", conf_matrix)



Test Accuracy : 99.95%
Precision     : 0.9991
Recall        : 1.0000
F1-Score      : 0.9995
Confusion Matrix:
 [[20165    24]
 [    1 26969]]


In [ ]:
def save_final_model(model, path="final_modelA.pth"):
    torch.save(model.state_dict(), path)
    print(f" Final model saved at {path}")

save_final_model(model)



 Final model saved at final_modelA.pth
